# **Task 7: Logistic Regression – Titanic Survival Prediction**


## **Importing Libraries and Loading Dataset:**

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('Datasets/Titanic-Dataset.csv')

In [ ]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## **Reviewing important features like Age,Sex,Fare,Survived:**

In [8]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [12]:
df['Embarked'].isnull().sum()

np.int64(2)

In [5]:
#review important features like Age, Sex, Fare, and Survived
df['Age'].isnull().sum()  # 177 null values

np.int64(177)

In [11]:
df['Survived'].value_counts() # These values are unevenly distributed

,count
Survived,
0,549
1,342


## **Replacing Age column null values with median**

In [17]:
median_age = df['Age'].describe().loc['50%']
median_age

np.float64(28.0)

In [18]:
filtered_dataset = df.copy()

In [20]:
filtered_dataset['Age'].isnull().sum()

np.int64(177)

In [21]:
filtered_dataset['Age'] = filtered_dataset['Age'].fillna(median_age)

In [22]:
filtered_dataset['Age'].isnull().sum()

np.int64(0)

## **Replacing Embarked column null values with mode value:**

In [34]:
mode_embarked = filtered_dataset['Embarked'].mode()[0]
mode_embarked

'S'

In [35]:
filtered_dataset['Embarked'].isnull().sum()

np.int64(2)

In [36]:
filtered_dataset['Embarked'] = filtered_dataset['Embarked'].fillna(mode_embarked)

In [37]:
filtered_dataset['Embarked'].isnull().sum()

np.int64(0)

## **Removing unnecessary columns like PassengerId or Name:**

In [38]:
filtered_dataset = filtered_dataset.drop(columns=['PassengerId','Name'])

In [39]:
filtered_dataset.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,0,3,male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,1,1,female,38.0,1,0,PC 17599,71.2833,C85,C
2,1,3,female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,1,1,female,35.0,1,0,113803,53.1000,C123,S
4,0,3,male,35.0,0,0,373450,8.0500,NaN,S


In [41]:
filtered_dataset['Embarked'].unique()

array(['S', 'C', 'Q'], dtype=object)

In [42]:
from sklearn.preprocessing import OneHotEncoder

In [51]:
ohe = OneHotEncoder(handle_unknown='ignore',sparse_output=False).set_output(transform="pandas")

In [52]:
encoded = ohe.fit_transform(filtered_dataset[['Embarked','Sex']])

In [54]:
encoded.head()

,Embarked_C,Embarked_Q,Embarked_S,Sex_female,Sex_male
0,0.0,0.0,1.0,0.0,1.0
1,1.0,0.0,0.0,1.0,0.0
2,0.0,0.0,1.0,1.0,0.0
3,0.0,0.0,1.0,1.0,0.0
4,0.0,0.0,1.0,0.0,1.0


In [55]:
filtered_dataset.drop(columns= ['Embarked','Sex'],inplace=True)
filtered_dataset.head()

,Survived,Pclass,Age,SibSp,Parch,Ticket,Fare,Cabin
0,0,3,22.0,1,0,A/5 21171,7.2500,NaN
1,1,1,38.0,1,0,PC 17599,71.2833,C85
2,1,3,26.0,0,0,STON/O2. 3101282,7.9250,NaN
3,1,1,35.0,1,0,113803,53.1000,C123
4,0,3,35.0,0,0,373450,8.0500,NaN


In [59]:
filtered_dataset = pd.concat([filtered_dataset,encoded],axis=1)
filtered_dataset.head()

,Survived,Pclass,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked_C,Embarked_Q,Embarked_S,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Sex_female,Sex_male
0,0,3,22.0,1,0,A/5 21171,7.2500,NaN,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
1,1,1,38.0,1,0,PC 17599,71.2833,C85,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0
2,1,3,26.0,0,0,STON/O2. 3101282,7.9250,NaN,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
3,1,1,35.0,1,0,113803,53.1000,C123,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
4,0,3,35.0,0,0,373450,8.0500,NaN,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0


In [63]:
from sklearn.preprocessing import StandardScaler

In [65]:
ss = StandardScaler().set_output(transform='pandas')

In [66]:
after_scaling = ss.fit_transform(filtered_dataset[['Age','Fare']])

In [68]:
after_scaling.head()

,Age,Fare
0,-0.565736,-0.502445
1,0.663861,0.786845
2,-0.258337,-0.488854
3,0.433312,0.420730
4,0.433312,-0.486337


In [70]:
filtered_dataset.drop(columns=['Age','Fare'],inplace=True)
filtered_dataset = pd.concat([filtered_dataset,after_scaling],axis=1)

In [71]:
filtered_dataset.head()

,Survived,Pclass,SibSp,Parch,Ticket,Cabin,Embarked_C,Embarked_Q,Embarked_S,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S,Sex_female,Sex_male,Age,Fare
0,0,3,1,0,A/5 21171,NaN,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,-0.565736,-0.502445
1,1,1,1,0,PC 17599,C85,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.663861,0.786845
2,1,3,0,0,STON/O2. 3101282,NaN,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,-0.258337,-0.488854
3,1,1,1,0,113803,C123,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.433312,0.420730
4,0,3,0,0,373450,NaN,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.433312,-0.486337


In [81]:
filtered_dataset.drop(columns=['Cabin','Ticket'],inplace=True)

In [82]:
from sklearn.model_selection import train_test_split

In [83]:
X = filtered_dataset.drop(columns = ['Survived'])
y = filtered_dataset['Survived']

In [84]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=24)

## **Train Logistic Regression model and generate predictions on test data:**

In [85]:
from sklearn.linear_model import LogisticRegression

In [86]:
model = LogisticRegression()

In [87]:
model.fit(X_train,y_train)

LogisticRegression()

In [88]:
y_pred = model.predict(X_test)

In [90]:
comparision = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred[:10]
})
comparision

,Actual,Predicted
0,0,0
1,1,1
2,0,0
3,0,0
4,1,1
5,1,1
6,1,1
7,1,0
8,0,0
9,0,0


## **Evaluate using accuracy, precision, recall, F1-score, and confusion matrix:**

In [91]:
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,confusion_matrix

In [99]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test,y_pred)

In [94]:
accuracy * 100

82.46268656716418

In [95]:
precision * 100

76.08695652173914

In [96]:
recall * 100

73.68421052631578

In [100]:
f1

0.7486631016042781

In [110]:
from sklearn.metrics import confusion_matrix

In [111]:
cm = confusion_matrix(y_test,y_pred)
cm

array([[151,  22],
       [ 25,  70]])

## **Plot ROC curve and calculate AUC score for more reliable evaluation:**

In [116]:
from sklearn.metrics import roc_curve, auc
y_prob = model.predict_proba(X_test)[:, 1]

# Calculate ROC values
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# Calculate AUC
roc_auc = auc(fpr, tpr)


In [117]:
roc_auc

np.float64(0.8739884393063583)